In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# データ読み込み
df = pd.read_csv('JAL_tripadvisor_reviews.csv')
ja_df = df[df['lang']=='ja'].copy()

print(f"総レビュー数: {len(df)}")
print(f"日本語レビュー数: {len(ja_df)}")
print(f"\n評価分布:")
print(ja_df['rating'].value_counts().sort_index())

: 

In [ ]:
# 形態素解析：日本語テキストを単語に分割（janomeを使用、品詞制限あり）

try:
    from janome.tokenizer import Tokenizer
    tokenizer = Tokenizer()
    print("janomeを使用します")
    JANOME_AVAILABLE = True
except ImportError:
    print("警告: janomeが見つかりません。簡易的な分かち書きを使用します")
    print("janomeをインストールするには: pip install janome")
    JANOME_AVAILABLE = False

# 抽出する品詞を指定（感情分析に有用な品詞のみ）
# 品詞の形式: "品詞,品詞細分類1,品詞細分類2,..."
ALLOWED_POS = [
    '名詞',      # 名詞全般（サービス、食事、座席など）
    '形容詞',    # 形容詞（良い、悪い、素晴らしいなど）
    '動詞',      # 動詞（満足する、感謝するなど）
    '副詞',      # 副詞（とても、非常になど）
]

def tokenize_japanese(text, allowed_pos=ALLOWED_POS):
    """日本語テキストの形態素解析（janomeを使用、品詞制限あり）"""
    if pd.isna(text) or not text:
        return []
    
    text = str(text)
    
    if JANOME_AVAILABLE:
        try:
            tokens = tokenizer.tokenize(text)
            filtered_tokens = []
            for token in tokens:
                # 品詞情報を取得（最初の要素が品詞）
                pos = token.part_of_speech.split(',')[0]
                # 指定した品詞のみを抽出
                if pos in allowed_pos:
                    surface = token.surface.strip()
                    if surface and len(surface) > 1:
                        filtered_tokens.append(surface)
            return filtered_tokens
        except:
            return []
    else:
        # 簡易的な分かち書き（スペースで分割）
        return [w for w in text.split() if len(w) > 1]

# 形態素解析を実行
ja_df['tokens'] = ja_df['text'].apply(tokenize_japanese)

print(f"形態素解析完了: {len(ja_df)}件のレビュー")
print(f"平均トークン数: {ja_df['tokens'].apply(len).mean():.1f}語")
print(f"\nサンプル（最初のレビュー）:")
if len(ja_df) > 0:
    print(f"  原文: {ja_df.iloc[0]['text'][:100]}...")
    print(f"  トークン: {ja_df.iloc[0]['tokens'][:20]}")


In [ ]:
# テキスト前処理：単語をIDに変換、シーケンス長の統一

import numpy as np
from collections import Counter
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import Tokenizer

# 語彙辞書の作成
tokenizer_keras = Tokenizer()
tokenizer_keras.fit_on_texts(ja_df['tokens'].apply(lambda x: ' '.join(x)))

# 単語をIDに変換
sequences = tokenizer_keras.texts_to_sequences(ja_df['tokens'].apply(lambda x: ' '.join(x)))

# シーケンスの長さを確認
seq_lengths = [len(seq) for seq in sequences]
max_length = int(np.percentile(seq_lengths, 95))  # 95パーセンタイルを使用
print(f"シーケンス長の統計:")
print(f"  平均: {np.mean(seq_lengths):.1f}")
print(f"  中央値: {np.median(seq_lengths):.1f}")
print(f"  最大: {max(seq_lengths)}")
print(f"  使用する最大長: {max_length}")

# パディング（シーケンス長を統一）
X = pad_sequences(sequences, maxlen=max_length, padding='post', truncating='post')

# 教師データの作成（ratingから感情ラベルを作成）
# rating >= 4: positive (1), rating <= 2: negative (0), その他: neutral (2)
def rating_to_label(rating):
    if rating >= 4:
        return 1  # positive
    elif rating <= 2:
        return 0  # negative
    else:
        return 2  # neutral

y = ja_df['rating'].apply(rating_to_label).values

# ラベルをone-hotエンコーディング
from tensorflow.keras.utils import to_categorical
y_categorical = to_categorical(y, num_classes=3)

print(f"\n前処理完了:")
print(f"  データ形状: {X.shape}")
print(f"  ラベル形状: {y_categorical.shape}")
print(f"  語彙サイズ: {len(tokenizer_keras.word_index) + 1}")
print(f"\nラベル分布:")
label_counts = Counter(y)
print(f"  negative (0): {label_counts[0]}件")
print(f"  positive (1): {label_counts[1]}件")
print(f"  neutral (2): {label_counts[2]}件")


In [ ]:
# LSTMモデルの構築

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.optimizers import Adam

vocab_size = len(tokenizer_keras.word_index) + 1
embedding_dim = 128
lstm_units = 64

# モデルの構築
model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=max_length),
    LSTM(lstm_units, dropout=0.2, recurrent_dropout=0.2),
    Dense(32, activation='relu'),
    Dropout(0.3),
    Dense(3, activation='softmax')  # 3クラス分類（negative, positive, neutral）
])

model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("LSTMモデル構築完了")
print("\nモデル構造:")
model.summary()


In [ ]:
# モデルの訓練（早期停止付き）

from sklearn.model_selection import train_test_split
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

# データを訓練用と検証用に分割
X_train, X_test, y_train, y_test = train_test_split(
    X, y_categorical, test_size=0.2, random_state=42, stratify=y
)

print(f"訓練データ: {X_train.shape[0]}件")
print(f"検証データ: {X_test.shape[0]}件")

# 早期停止の設定
early_stopping = EarlyStopping(
    monitor='val_loss',        # 監視する指標（検証損失）
    patience=3,                # 改善が見られないエポック数を何回まで許容するか
    restore_best_weights=True, # 最良の重みを復元
    verbose=1                  # 停止時にメッセージを表示
)

# モデルの訓練
history = model.fit(
    X_train, y_train,
    batch_size=32,
    epochs=50,                 # 最大エポック数を増やす（早期停止で自動的に停止）
    validation_data=(X_test, y_test),
    callbacks=[early_stopping], # 早期停止コールバックを追加
    verbose=1
)

print("\n訓練完了")
print(f"実際の訓練エポック数: {len(history.history['loss'])}")


In [ ]:
# 予測と評価

# 全データで予測
y_pred_proba = model.predict(X)
y_pred = np.argmax(y_pred_proba, axis=1)

# 予測結果をDataFrameに追加
ja_df['lstm_sentiment_label'] = y_pred
ja_df['lstm_sentiment_proba'] = y_pred_proba.max(axis=1)

# ラベル名のマッピング
label_map = {0: 'negative', 1: 'positive', 2: 'neutral'}
ja_df['lstm_sentiment_label_name'] = ja_df['lstm_sentiment_label'].map(label_map)

# 精度評価
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

accuracy = accuracy_score(y, y_pred)
print(f"全体精度: {accuracy:.3f}")

print("\n分類レポート:")
print(classification_report(y, y_pred, target_names=['negative', 'positive', 'neutral']))

print("\n混同行列:")
print(confusion_matrix(y, y_pred))

# 予測結果の分布
print(f"\n予測ラベル分布:")
print(ja_df['lstm_sentiment_label_name'].value_counts())


In [ ]:
# 可視化

fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# 1. 評価分布
axes[0, 0].bar(ja_df['rating'].value_counts().sort_index().index, 
               ja_df['rating'].value_counts().sort_index().values,
               color='skyblue', alpha=0.7)
axes[0, 0].set_xlabel('Rating')
axes[0, 0].set_ylabel('Count')
axes[0, 0].set_title('Rating Distribution')
axes[0, 0].grid(True, alpha=0.3)

# 2. LSTM予測ラベル分布
sentiment_counts = ja_df['lstm_sentiment_label_name'].value_counts()
axes[0, 1].bar(sentiment_counts.index, sentiment_counts.values,
               color=['red', 'green', 'gray'], alpha=0.7)
axes[0, 1].set_xlabel('Sentiment Label')
axes[0, 1].set_ylabel('Count')
axes[0, 1].set_title('LSTM Sentiment Label Distribution')
axes[0, 1].grid(True, alpha=0.3)

# 3. 訓練履歴（精度）
axes[1, 0].plot(history.history['accuracy'], label='Train Accuracy', marker='o')
axes[1, 0].plot(history.history['val_accuracy'], label='Validation Accuracy', marker='s')
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('Accuracy')
axes[1, 0].set_title('Model Training History (Accuracy)')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# 4. 訓練履歴（損失）
axes[1, 1].plot(history.history['loss'], label='Train Loss', marker='o')
axes[1, 1].plot(history.history['val_loss'], label='Validation Loss', marker='s')
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('Loss')
axes[1, 1].set_title('Model Training History (Loss)')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
# サマリー

print("=" * 60)
print("LSTMによる感情分析サマリー")
print("=" * 60)

print(f"\n【基本統計】")
print(f"  総レビュー数: {len(ja_df):,}件")
print(f"  平均評価: {ja_df['rating'].mean():.2f}")

print(f"\n【評価分布】")
rating_dist = ja_df['rating'].value_counts().sort_index()
for rating, count in rating_dist.items():
    percentage = (count / len(ja_df)) * 100
    print(f"  {rating}つ星: {count:,}件 ({percentage:.1f}%)")

print(f"\n【LSTM感情分析結果】")
sentiment_dist = ja_df['lstm_sentiment_label_name'].value_counts()
for label, count in sentiment_dist.items():
    percentage = (count / len(ja_df)) * 100
    print(f"  {label}: {count:,}件 ({percentage:.1f}%)")

print(f"\n【モデル性能】")
print(f"  全体精度: {accuracy:.3f}")
print(f"  平均予測確信度: {ja_df['lstm_sentiment_proba'].mean():.3f}")

# ratingとの一致率を計算
true_labels = ja_df['rating'].apply(rating_to_label)
match_rate = (true_labels == ja_df['lstm_sentiment_label']).mean()
print(f"  ratingとの一致率: {match_rate:.3f}")

print("\n" + "=" * 60)
